In [ ]:
# ── Validate Feast version ───────────────────────────────────────────
import os
import feast

actual_version = feast.__version__
expected_version = os.environ.get("FEAST_VERSION")

assert actual_version == expected_version, (
    f"❌ Feast version mismatch. Expected: {expected_version}, Found: {actual_version}"
)
print(f"✅ Feast version validated: {actual_version}")

In [ ]:
# ──  Navigate to feature repo ─────────────────────────────────────────
%cd /opt/app-root/src/feature_repo
!ls -l

In [ ]:
# ── Validate feature_store.yaml ──────────────────────────────────────
import yaml

with open("feature_store.yaml") as f:
    fs_config = yaml.safe_load(f)

print(yaml.dump(fs_config, default_flow_style=False))

assert fs_config.get("project") == "rag", (
    f"❌ Expected project 'rag', got '{fs_config.get('project')}'"
)
online_store = str(fs_config.get("online_store", "")).lower()
assert "milvus" in online_store, (
    f"❌ Milvus not configured as online store. Found: {fs_config.get('online_store')}"
)
print("✅ feature_store.yaml validated: project=rag, online_store=milvus")

In [ ]:
# ── Download dataset ──────────────────────────────────────────────────
!mkdir -p data
!wget -q -O data/city_wikipedia_summaries_with_embeddings.parquet \
    https://raw.githubusercontent.com/opendatahub-io/feast/master/examples/rag/feature_repo/data/city_wikipedia_summaries_with_embeddings.parquet
print("✅ Dataset downloaded")

In [ ]:
#  Load dataset and validate embeddings ──────────────────────────────
import pandas as pd

df = pd.read_parquet("./data/city_wikipedia_summaries_with_embeddings.parquet")
df["vector"] = df["vector"].apply(lambda x: x.tolist())

assert not df.empty, "❌ Loaded dataframe is empty"

required_columns = {"item_id", "vector", "state", "sentence_chunks", "wiki_summary"}
missing = required_columns - set(df.columns)
assert not missing, f"❌ Missing columns in parquet: {missing}"

embedding_length = len(df["vector"][0])
assert embedding_length == 384, f"❌ Expected vector length 384, got {embedding_length}"

print(f"✅ Dataset loaded: {len(df)} rows, embedding length={embedding_length}")
df.head()

In [ ]:
# Install dependencies ──────────────────────────────────────────────
!pip install -q pymilvus[milvus_lite] transformers torch
print("✅ Dependencies installed")

In [ ]:
#  Run feast apply and validate output ───────────────────────────────
import subprocess

result = subprocess.run(["feast", "apply"], capture_output=True, text=True)
output = result.stdout + result.stderr
print(output)

assert result.returncode == 0, (
    f"❌ feast apply exited with code {result.returncode}\n{output}"
)

expected_messages = [
    "Applying changes for project rag",
    "Deploying infrastructure for city_embeddings",
]
for msg in expected_messages:
    assert msg in output, f"❌ Expected message not found in feast apply output: '{msg}'"

print("✅ feast apply succeeded with all expected messages")

In [ ]:
#  Initialize FeatureStore ──────────────────────────────────────────
from feast import FeatureStore

store = FeatureStore(repo_path=".")
print(f"✅ FeatureStore initialized — project: {store.project}")

In [ ]:
try:
    store.write_to_online_store(feature_view_name="city_embeddings", df=df)
    print("✅ write_to_online_store executed successfully")
except Exception as e:
    raise AssertionError(f"❌ write_to_online_store failed: {e}")

# Verify row count in Milvus matches source dataframe
milvus_client = store._provider._online_store._connect(store.config)
collections = milvus_client.list_collections()
assert len(collections) > 0, "❌ No Milvus collections found after write_to_online_store"

actual_count = milvus_client.query(
    collection_name=collections[0],
    filter="",
    output_fields=["count(*)"],
)[0].get("count(*)", 0)

assert actual_count == len(df), (
    f"❌ Row count mismatch: wrote {len(df)}, Milvus has {actual_count}"
)
print(f"✅ Milvus contains all {actual_count} written rows")

In [ ]:
batch_fvs = store.list_batch_feature_views()

assert isinstance(batch_fvs, list), "❌ list_batch_feature_views did not return a list"
assert len(batch_fvs) == 1, (
    f"❌ Expected exactly 1 batch feature view, got {len(batch_fvs)}"
)
print(f"✅ Batch feature views count validated: {len(batch_fvs)}")

In [ ]:
fv = store.get_feature_view("city_embeddings")

assert fv.name == "city_embeddings", (
    f"❌ Feature view name mismatch: expected 'city_embeddings', got '{fv.name}'"
)
assert fv.entities == ["item_id"], (
    f"❌ Expected entities ['item_id'], got {fv.entities}"
)

feature_names = [f.name for f in fv.features]
for expected_field in ["vector", "state", "sentence_chunks", "wiki_summary"]:
    assert expected_field in feature_names, (
        f"❌ Missing expected feature field: '{expected_field}'"
    )

vector_feature = next(f for f in fv.features if f.name == "vector")
assert vector_feature.vector_index, "❌ 'vector' feature is not configured as a vector index"
assert vector_feature.vector_search_metric == "COSINE", (
    f"❌ Expected COSINE search metric, got '{vector_feature.vector_search_metric}'"
)

print(f"✅ Feature view 'city_embeddings' schema validated")
print(f"   Fields   : {feature_names}")
print(f"   Entities : {fv.entities}")
print(f"   Metric   : {vector_feature.vector_search_metric}")

In [ ]:
from feast.entity import Entity
from feast.types import ValueType

test_entity = Entity(
    name="item_id1",
    value_type=ValueType.INT64,
    description="Temporary test entity",
    tags={"team": "feast"},
)
store.apply(test_entity)

assert any(e.name == "item_id1" for e in store.list_entities()), (
    "❌ Entity 'item_id1' not found after apply"
)
print("✅ Entity 'item_id1' added and verified")

In [ ]:
entity_to_delete = store.get_entity("item_id1")
store.apply(objects=[], objects_to_delete=[entity_to_delete], partial=False)

assert not any(e.name == "item_id1" for e in store.list_entities()), (
    "❌ Entity 'item_id1' still present after deletion"
)
print("✅ Entity 'item_id1' deleted and verified")

In [ ]:
batch_fvs = store.list_batch_feature_views()
assert len(batch_fvs) == 1, (
    f"❌ Expected 1 feature view after entity delete, got {len(batch_fvs)}"
)
print(f"✅ Feature view count unchanged after entity delete: {len(batch_fvs)}")

In [ ]:

milvus_client = store._provider._online_store._connect(store.config)

collections = milvus_client.list_collections()
assert len(collections) > 0, "❌ No Milvus collections found"
COLLECTION_NAME = collections[0]
print(f"Collection: {COLLECTION_NAME}")

milvus_results = milvus_client.query(
    collection_name=COLLECTION_NAME,
    filter="item_id == '0'",
)
assert len(milvus_results) > 0, "❌ Milvus query for item_id=0 returned no results"

result_df = pd.DataFrame(milvus_results)
assert "vector" in result_df.columns, "❌ 'vector' column missing from Milvus query result"

print(f"✅ Milvus direct query validated: {len(result_df)} row(s) returned")
result_df.head()

In [ ]:

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

TOKENIZER = "sentence-transformers/all-MiniLM-L6-v2"
MODEL = "sentence-transformers/all-MiniLM-L6-v2"

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = (
        attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    )
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
        input_mask_expanded.sum(1), min=1e-9
    )

def run_model(sentences, tokenizer, model):
    encoded_input = tokenizer(
        sentences, padding=True, truncation=True, return_tensors="pt"
    )
    with torch.no_grad():
        model_output = model(**encoded_input)
    embeddings = mean_pooling(model_output, encoded_input["attention_mask"])
    return F.normalize(embeddings, p=2, dim=1)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER)
model = AutoModel.from_pretrained(MODEL)
print("✅ Embedding model loaded")

In [ ]:
# ── Cell 17: Generate query embedding ─────────────────────────────────────────
question = "Which city has the largest population in New York?"

query_embedding = run_model(question, tokenizer, model)
query = query_embedding.detach().cpu().numpy().tolist()[0]

assert len(query) == 384, f"❌ Query embedding length mismatch: expected 384, got {len(query)}"
print(f"✅ Query embedding generated: length={len(query)}")

In [ ]:

TOP_K = 3

context_data = store.retrieve_online_documents_v2(
    features=[
        "city_embeddings:vector",
        "city_embeddings:item_id",
        "city_embeddings:state",
        "city_embeddings:sentence_chunks",
        "city_embeddings:wiki_summary",
    ],
    query=query,
    top_k=TOP_K,
    distance_metric="COSINE",
).to_df()

assert context_data is not None and not context_data.empty, (
    "❌ retrieve_online_documents_v2 returned empty results"
)
assert len(context_data) == TOP_K, (
    f"❌ Expected top_k={TOP_K} results, got {len(context_data)}"
)

expected_columns = {"state", "sentence_chunks", "wiki_summary", "item_id", "vector"}
missing_cols = expected_columns - set(context_data.columns)
assert not missing_cols, f"❌ Missing columns in RAG result: {missing_cols}"

if "distance" in context_data.columns:
    assert context_data["distance"].between(-1.0, 1.0).all(), (
        "❌ COSINE distance scores out of expected range [-1, 1]"
    )
    print(f"   Distance range: [{context_data['distance'].min():.4f}, {context_data['distance'].max():.4f}]")

print(f"✅ retrieve_online_documents_v2 returned {len(context_data)} results with correct schema")
context_data

In [ ]:
def format_documents(context_df):
    output_context = ""
    unique_documents = context_df.drop_duplicates(subset=["item_id"]).apply(
        lambda x: "City & State = {" + x["state"] + "}\nSummary = {" + x["wiki_summary"].strip() + "}",
        axis=1,
    )
    for i, document_text in enumerate(unique_documents):
        output_context += f"****START DOCUMENT {i}****\n{document_text.strip()}\n****END DOCUMENT {i}****\n"
    return output_context

# Pass item_id so drop_duplicates(subset=["item_id"]) has the column available
RAG_CONTEXT = format_documents(context_data[["item_id", "state", "wiki_summary"]])

assert RAG_CONTEXT.strip(), "❌ format_documents returned empty context"
assert "START DOCUMENT" in RAG_CONTEXT, "❌ START DOCUMENT marker missing from RAG context"
assert "END DOCUMENT" in RAG_CONTEXT, "❌ END DOCUMENT marker missing from RAG context"
assert RAG_CONTEXT.count("START DOCUMENT") == TOP_K, (
    f"❌ Expected {TOP_K} documents in context, found {RAG_CONTEXT.count('START DOCUMENT')}"
)

print(f"✅ RAG context formatted correctly: {TOP_K} documents")
print(RAG_CONTEXT)

In [ ]:

FULL_PROMPT = f"""
You are an assistant for answering questions about states. You will be provided documentation
from Wikipedia. Provide a conversational answer. If you don't know the answer, just say
"I do not know." Don't make up an answer.

Here are document(s) you should use when answering the user's question:
{RAG_CONTEXT}
"""

assert RAG_CONTEXT in FULL_PROMPT, "❌ RAG context not embedded in prompt"
print("✅ Full prompt constructed")
print(FULL_PROMPT)